# 03 — Deploy `contoso-poc-model` to a managed online endpoint

This notebook drives the **proven** end-to-end flow for a managed online
endpoint on a UAMI-only AML workspace. It mirrors the helper scripts in
`src/` so each step is reproducible from either the notebook or the CLI.

**Flow:**

1. Patch workspace `systemDatastoresAuthMode=identity` (one-time).
2. Create `contoso-endpoint` with `auth_mode='aml_token'` and the default
   system-assigned MI (do **not** pass an explicit UAMI — see docs/06
   Issue 4.9).
3. Grant the endpoint MI + workspace UAMI the three required RBAC roles
   (Issues 4.10 + 4.11).
4. Patch the registered MLflow model to add `azureml-ai-monitoring` to
   its environment (Issue 4.12), re-register as a new version.
5. **Blue/green deploy** the patched model: create the inactive color,
   shift 100% traffic, then delete the old color (zero downtime).
6. Smoke-test via traffic routing to prove the shift took effect
   (Issue 4.13 payload format).

> Pre-req: `contoso-poc-model:1` already registered (notebook 02 done).

In [ ]:
import sys
sys.path.insert(0, "..")  # so `from src.config import load_settings` works

ENDPOINT_NAME    = "contoso-endpoint"
MODEL_NAME       = "contoso-poc-model"
SOURCE_VERSION   = "1"            # version to patch
INSTANCE_TYPE    = "Standard_DS2_v2"
INSTANCE_COUNT   = 1
SAMPLE_REQUEST   = "../data/sample_request.json"

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.config import load_settings

s = load_settings()
ml = MLClient(DefaultAzureCredential(), s.subscription_id, s.resource_group, s.aml_workspace)
print("Workspace:", s.aml_workspace)
print("RG:       ", s.resource_group)

## 1. Patch workspace to `systemDatastoresAuthMode=identity` (one-time)

Required for UAMI-only workspaces so AML uses identity (not access keys)
to talk to the workspace storage. Skip if already patched.

In [ ]:
import subprocess, json
url = (
    f"https://management.azure.com/subscriptions/{s.subscription_id}"
    f"/resourceGroups/{s.resource_group}/providers/Microsoft.MachineLearningServices"
    f"/workspaces/{s.aml_workspace}?api-version=2024-04-01"
)
out = subprocess.check_output(
    ["az", "rest", "--method", "GET", "--url", url], text=True
)
mode = json.loads(out)["properties"].get("systemDatastoresAuthMode")
print("Current systemDatastoresAuthMode:", mode)

if mode != "identity":
    subprocess.run(
        [
            "az", "rest", "--method", "PATCH", "--url", url,
            "--body", '{"properties":{"systemDatastoresAuthMode":"identity"}}',
        ],
        check=True,
    )
    print("Patched -> identity")
else:
    print("Already identity, skipping.")

## 2. Create the endpoint (system-assigned MI, aml_token auth)

> **Do not** pass `IdentityConfiguration(type='user_assigned', ...)`.
> On UAMI-only workspaces this returns an opaque `InternalServerError`
> 500. Let AML default to a system-assigned MI (Issue 4.9).

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

try:
    ep = ml.online_endpoints.get(ENDPOINT_NAME)
    print(f"Endpoint exists: {ep.name} ({ep.provisioning_state})")
except Exception:
    print(f"Creating {ENDPOINT_NAME}...")
    ml.online_endpoints.begin_create_or_update(
        ManagedOnlineEndpoint(name=ENDPOINT_NAME, auth_mode="aml_token")
    ).result()
    ep = ml.online_endpoints.get(ENDPOINT_NAME)

print("State:        ", ep.provisioning_state)
print("Auth:         ", ep.auth_mode)
print("Scoring URI:  ", ep.scoring_uri)
print("MI principal: ", ep.identity.principal_id)

## 3. Grant required RBAC

Three grants are required before any deployment can succeed:

| # | Principal | Role | Scope |
|---|---|---|---|
| 1 | Endpoint system MI | `Storage Blob Data Reader` | Workspace storage account |
| 2 | Endpoint system MI | `AcrPull` | Workspace ACR |
| 3 | Workspace **UAMI** | `Key Vault Secrets Officer` | Workspace KV (Secrets User is **not** enough — Issue 4.11) |

Uses the `grant-rbac` helper in `src/deploy_online_endpoint.py`.

In [ ]:
import subprocess
subprocess.run(
    ["uv", "run", "python", "-m", "src.deploy_online_endpoint", "grant-rbac"],
    cwd="..",
    check=True,
)
print("Done. Wait ~30-60s for RBAC propagation before deploying.")

## 4. Patch the MLflow model env — add `azureml-ai-monitoring`

The auto-generated MLflow scoring script imports
`azureml.ai.monitoring.Collector`. If your training run didn't log this
package via `mlflow.sklearn.log_model(..., pip_requirements=[...])`, the
deployment container will crash with `ModuleNotFoundError: azureml`
(Issue 4.12).

This step downloads `contoso-poc-model:SOURCE_VERSION`, patches its
`conda.yaml` + `requirements.txt`, and registers a new version.

> Skip this cell if you already have a version with `azureml-ai-monitoring`.

In [ ]:
import subprocess
subprocess.run(
    [
        "uv", "run", "python", "-m", "src.patch_mlflow_model",
        "--name", MODEL_NAME,
        "--version", SOURCE_VERSION,
        "--add-package", "azureml-ai-monitoring",
        "--description", f"v2: added azureml-ai-monitoring (patched from v{SOURCE_VERSION})",
    ],
    cwd="..",
    check=True,
)

# Discover the latest version after patching
versions = sorted(
    ml.models.list(name=MODEL_NAME),
    key=lambda m: int(m.version),
)
DEPLOY_VERSION = versions[-1].version
print(f"Latest model version: {MODEL_NAME}:{DEPLOY_VERSION}")

## 5. Blue/green deploy — zero-downtime traffic shift

The `deploy` helper implements the standard **blue/green** pattern:

1. Picks `target` = whichever color is **not** currently serving traffic
   (first run ever → `blue`).
2. Cleans up any stale `target` deployment from a prior failed attempt.
3. Creates the new `target` deployment from `MODEL_NAME:DEPLOY_VERSION`.
4. Shifts 100% traffic from the live color to `target`.
5. Deletes the previous color (now at 0% traffic).

The previously-live deployment keeps serving until step 4 completes, so
callers see **no downtime**. Image build ~10 min + container start & traffic allocation ~10 min
on `Standard_DS2_v2`. Watch progress at **AML Studio → Endpoints →
contoso-endpoint → Deployments**.

In [ ]:
import subprocess
subprocess.run(
    [
        "uv", "run", "python", "-m", "src.deploy_online_endpoint",
        "deploy", "--model-version", str(DEPLOY_VERSION),
    ],
    cwd="..",
    check=True,
)

## 6. Smoke test

Build a 3-row sample request from the local snapshot, then invoke the
endpoint **without pinning a deployment** — the request follows the
endpoint's traffic weights, which proves the blue/green shift in step 5
actually took effect.

AML's MLflow wrapper requires the **`input_data`** payload shape with
all three of `columns`, `index`, and `data` (Issue 4.13).

Expected response: `["True", "True", "True"]`.

In [ ]:
import subprocess, json
subprocess.run(
    [
        "uv", "run", "python", "-m", "src.build_sample_request",
        "--rows", "3",
        "--output", "data/sample_request.json",
    ],
    cwd="..",
    check=True,
)

# No deployment_name -> request follows endpoint traffic weights.
response = ml.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    request_file=SAMPLE_REQUEST,
)
print("Response:", response)
try:
    print("Parsed: ", json.loads(response))
except Exception:
    pass

## Final status

In [ ]:
ep = ml.online_endpoints.get(ENDPOINT_NAME)
print("Endpoint:", ep.name, "|", ep.provisioning_state, "| traffic:", ep.traffic)
print("Scoring URI:", ep.scoring_uri)
for d in ml.online_deployments.list(endpoint_name=ENDPOINT_NAME):
    print(f"  - {d.name} | {d.provisioning_state} | sku={d.instance_type} | model={d.model}")